In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20220621185332Bh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20111208173814Mh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20200806072410Th.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20111017113714Th.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20170926225530Lh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20160327133734Ch.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20160820233134Lh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images/20130310063154Uh.jpeg
/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.

In [17]:
import os
import sys
import json
import time
import warnings
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Install segmentation_models_pytorch if not present
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("Installing segmentation-models-pytorch...")
    os.system("pip install -q segmentation-models-pytorch albumentations")
    import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Global Configuration
# CFG = {
#     'seed': 42,
#     'img_size': (512, 512),
#     'batch_size': 8,
#     'epochs': 2,#35
#     'lr': 3e-3,
#     'backbone': 'resnet34',
#     'encoder_weights': 'imagenet',
#     'num_workers': 2,
#     'device': 'cuda' if torch.cuda.is_available() else 'cpu',
#     'threshold': 0.5
# }

CFG = {
    'seed': 43,
    'img_size': (512, 512),
    'batch_size': 16,
    'epochs': 35,
    'lr': 3e-3,
    'backbone': 'resnet34',
    'encoder_weights': 'imagenet',
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'threshold': 0.5
}

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
print(f"Using Device: {CFG['device']}")


Using Device: cuda


In [18]:
DATA_DIR = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/")
TRAIN_DIR = DATA_DIR / "train/train_images"
TEST_DIR = DATA_DIR / "test/test_images"
LABELS_PATH = DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

# Load Annotations
with open(LABELS_PATH, 'r') as f:
    labels_data = json.load(f)

# Get list of images
df_train = pd.DataFrame({'filename': sorted([p.name for p in TRAIN_DIR.glob("*.jpeg")])})
df_test  = pd.DataFrame({'filename': sorted([p.name for p in TEST_DIR.glob("*.jpeg")])})

# Clean & Stratified-Ready Train / Validation Split (80/20)
train_df, val_df = train_test_split(
    df_train, 
    test_size=0.2, 
    random_state=CFG['seed'], 
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train Images: {len(train_df)} \nVal   Images: {len(val_df)}\nTest  Images: {len(df_test)}")


Train Images: 565 
Val   Images: 142
Test  Images: 180


In [19]:
def create_mask_from_annotation(ann_entry, img_shape):
    """
    Generates a binary mask array from JSON annotations.
    Supports COCO polygon lists, point sets, or direct contour coordinates.
    """
    h, w = img_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    if not ann_entry:
        return mask
        
    try:
        # Handling polygon coordinates
        if isinstance(ann_entry, list):
            for poly in ann_entry:
                if isinstance(poly, list) and len(poly) > 0:
                    pts = np.array(poly, dtype=np.int32)
                    if pts.ndim == 1:
                        pts = pts.reshape((-1, 2))
                    cv2.fillPoly(mask, [pts], 1)
        elif isinstance(ann_entry, dict) and 'segmentation' in ann_entry:
            for poly in ann_entry['segmentation']:
                pts = np.array(poly, dtype=np.int32).reshape((-1, 2))
                cv2.fillPoly(mask, [pts], 1)
    except Exception:
        # Fallback empty mask on formatting anomaly
        pass
        
    return mask

class SolarDataset(Dataset):
    def __init__(self, df, img_dir, annotations=None, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.annotations = annotations
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx]['filename']
        img_path = os.path.join(self.img_dir, filename)
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape
        
        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, filename
            
        # Get target mask
        ann_entry = self.annotations.get(filename, []) if self.annotations else []
        mask = create_mask_from_annotation(ann_entry, (h, w))
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        mask = mask.unsqueeze(0).float()
        return image, mask

In [20]:
# Augmentations using Albumentations
train_transform = A.Compose([
    A.Resize(CFG['img_size'][0], CFG['img_size'][1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(CFG['img_size'][0], CFG['img_size'][1]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [21]:
# Create U-Net Model
def build_model():
    model = smp.Unet(
        encoder_name=CFG['backbone'],
        encoder_weights=CFG['encoder_weights'],
        in_channels=3,
        classes=1,
        activation=None  # Output logits
    )
    return model

# Hybrid Loss Function: Dice Loss + BCE Loss
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, y_pred, y_true):
        return 0.5 * self.bce(y_pred, y_true) + 0.5 * self.dice(y_pred, y_true)

model = build_model().to(CFG['device'])
criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])

print(f"Model built with backbone '{CFG['backbone']}'")


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Model built with backbone 'resnet34'


In [22]:
# Create DataLoaders
train_dataset = SolarDataset(train_df, TRAIN_DIR, labels_data, transform=train_transform)
val_dataset = SolarDataset(val_df, TRAIN_DIR, labels_data, transform=val_transform)

train_dataloader = DataLoader(train_dataset, 
                              batch_size=CFG['batch_size'], 
                              shuffle=True, 
                              num_workers=CFG['num_workers'])

val_dataloader = DataLoader(val_dataset, 
                            batch_size=CFG['batch_size'], 
                            shuffle=False, 
                            num_workers=CFG['num_workers'])

In [23]:
best_val_loss = float('inf')

for epoch in range(CFG['epochs']):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, masks in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Train]"):
        images, masks = images.to(CFG['device']), masks.to(CFG['device']) # tranfser to device
        
        optimizer.zero_grad()                #zero gradients
        outputs = model(images)              #forward pass
        loss = criterion(outputs, masks)     #loss calculation
        loss.backward()                      #backward pass
        optimizer.step()                     #update model params
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_dataloader)
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Val]"):
            images, masks = images.to(CFG['device']), masks.to(CFG['device'])
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_dataloader)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_unet_model.pth")
        print("Best model saved!")


Epoch 1/35 [Val]: 100%|██████████| 9/9 [00:03<00:00,  2.54it/s]


Epoch 01 | Train Loss: 0.0515 | Val Loss: 0.0035
Best model saved!


Epoch 2/35 [Val]: 100%|██████████| 9/9 [00:03<00:00,  2.41it/s]


Epoch 02 | Train Loss: 0.0021 | Val Loss: 0.0014
Best model saved!


Epoch 3/35 [Val]: 100%|██████████| 9/9 [00:03<00:00,  2.58it/s]


Epoch 03 | Train Loss: 0.0011 | Val Loss: 0.0009
Best model saved!


... (60 outputs left out)

Epoch 34/35 [Val]: 100%|██████████| 9/9 [00:03<00:00,  2.53it/s]


Epoch 34 | Train Loss: 0.0000 | Val Loss: 0.0000
Best model saved!


Epoch 35/35 [Val]: 100%|██████████| 9/9 [00:03<00:00,  2.61it/s]


Epoch 35 | Train Loss: 0.0000 | Val Loss: 0.0000
Best model saved!


In [24]:
import pycocotools.mask as mask_utils

def mask_to_coco_rle(binary_mask):
    """
    Encodes a binary numpy mask (H, W) into COCO compressed RLE ASCII string.
    """
    # pycocotools expects Fortran-contiguous array (column-major)
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    # Convert bytes to string for CSV
    rle['counts'] = rle['counts'].decode('utf-8')
    return rle['counts']

# Load best trained model weights
model.load_state_dict(torch.load("best_unet_model.pth"))
model.eval()

test_dataset = SolarDataset(df_test, TEST_DIR, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

submissions = []

print("🔮 Generating Compressed COCO RLE Predictions for Test Images...")

with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(CFG['device'])
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()[0, 0]
        
        # Binary thresholding
        binary_mask = (probs > CFG['threshold']).astype(np.uint8)
        
        image_name = Path(filenames[0]).stem  # e.g., '20110120105534Ch'
        
        # Find individual connected filament components
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask)
        
        filament_count = 0
        for label in range(1, num_labels):  # Skip background (label 0)
            # Filter out tiny noise artifacts
            if stats[label, cv2.CC_STAT_AREA] < 30: 
                continue
                
            single_filament_mask = (labels == label).astype(np.uint8)
            
            # Encode into COCO ASCII RLE format
            coco_rle_str = mask_to_coco_rle(single_filament_mask)
            
            if coco_rle_str:
                filament_count += 1
                submissions.append({
                    'filament_id': f"{image_name}_{filament_count}",
                    'segmentation_rle': coco_rle_str
                })
        
        # Fallback: If no filaments detected, encode an empty background mask
        if filament_count == 0:
            empty_mask = np.zeros(binary_mask.shape, dtype=np.uint8)
            empty_rle_str = mask_to_coco_rle(empty_mask)
            submissions.append({
                'filament_id': f"{image_name}_1",
                'segmentation_rle': empty_rle_str
            })

# Construct Submission DataFrame
sub_df = pd.DataFrame(submissions)

# Verify Output
print("\n🔍 Verification Check for NULL / NaN values:")
print(sub_df.isnull().sum())

print(f"\n📊 Total predictions generated: {len(sub_df)}")

# Save to CSV
sub_df.to_csv("submission.csv", index=False)
print("🎉 submission.csv generated successfully with COCO Compressed RLE strings!")
print(sub_df.head(10))


🔮 Generating Compressed COCO RLE Predictions for Test Images...


100%|██████████| 180/180 [00:09<00:00, 19.85it/s]


🔍 Verification Check for NULL / NaN values:
filament_id         0
segmentation_rle    0
dtype: int64

📊 Total predictions generated: 180
🎉 submission.csv generated successfully with COCO Compressed RLE strings!
          filament_id segmentation_rle
0  20110120105534Ch_1             PPP8
1  20110130110334Ch_1             PPP8
2  20110214175414Mh_1             PPP8
3  20110329082654Uh_1             PPP8
4  20110408083214Th_1             PPP8
5  20110424064534Lh_1             PPP8
6  20110429120534Ch_1             PPP8
7  20110529063154Uh_1             PPP8
8  20110531070514Th_1             PPP8
9  20110603123034Ch_1             PPP8
